# SPY Hybrid ERNN A Self-lagged

Original unrestricted model tuning with seven separately estimated nominal levels and the Bitcoin rank-based selection procedure.

In [2]:
# ============================================================
# SPY MODELLING DATA PREPARATION
# Common setup for all ERNN and QRNN specifications
# ============================================================

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch


# ------------------------------------------------------------
# 1. Reproducibility and device
# ------------------------------------------------------------

BASE_SEED = 2026

random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. Load the scaled SPY dataset
# ------------------------------------------------------------

DATA_PATH = Path.home() / "SPY_features_scaled.csv"

df = pd.read_csv(
    DATA_PATH,
    index_col="Date",
    parse_dates=True
)

df = (
    df
    .sort_index()
    .drop_duplicates()
    .dropna()
    .copy()
)


# ------------------------------------------------------------
# 3. Response and predictors
# ------------------------------------------------------------

target_col = "log_return"

feature_cols = [
    "Volatility",
    "RSI_14",
    "ATR_14",
    "QQQ_LogReturns",
    "VIX_LogChange",
    "Volume_LogChange",
]

required_cols = [target_col] + feature_cols

missing_cols = [
    column
    for column in required_cols
    if column not in df.columns
]

if missing_cols:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_cols)
    )


# ------------------------------------------------------------
# 4. Chronological 70%-15%-15% split
# ------------------------------------------------------------

train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size:].copy()


# ------------------------------------------------------------
# 5. Convert samples to tensors
# ------------------------------------------------------------

def dataframe_to_tensors(sample):
    X = torch.tensor(
        sample[feature_cols].to_numpy(),
        dtype=torch.float32,
        device=device
    )
    y = torch.tensor(
        sample[target_col].to_numpy(),
        dtype=torch.float32,
        device=device
    )
    return X, y


X_train, y_train = dataframe_to_tensors(train_df)
X_val, y_val = dataframe_to_tensors(val_df)
X_test, y_test = dataframe_to_tensors(test_df)

# Compatibility aliases used by later QRNN cells.
X_test_q = X_test
y_test_q = y_test

levels = [
    0.025,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.975,
]

alpha_levels = levels
tau_levels = levels


# ------------------------------------------------------------
# 6. Checks and split summary
# ------------------------------------------------------------

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
assert X_train.shape[1] == len(feature_cols)
assert torch.isfinite(X_train).all()
assert torch.isfinite(X_val).all()
assert torch.isfinite(X_test).all()
assert torch.isfinite(y_train).all()
assert torch.isfinite(y_val).all()
assert torch.isfinite(y_test).all()

split_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Observations": [len(train_df), len(val_df), len(test_df)],
    "Start": [
        train_df.index.min().date(),
        val_df.index.min().date(),
        test_df.index.min().date(),
    ],
    "End": [
        train_df.index.max().date(),
        val_df.index.max().date(),
        test_df.index.max().date(),
    ],
})

display(split_summary)

print("Input features:", feature_cols)
print("Number of predictors:", len(feature_cols))
print("X_train shape:", tuple(X_train.shape))
print("X_val shape:  ", tuple(X_val.shape))
print("X_test shape: ", tuple(X_test.shape))


Device: cpu


,Sample,Observations,Start,End
0,Training,2219,2014-01-23,2022-11-11
1,Validation,475,2022-11-14,2024-10-04
2,Test,476,2024-10-07,2026-08-31


Input features: ['Volatility', 'RSI_14', 'ATR_14', 'QQQ_LogReturns', 'VIX_LogChange', 'Volume_LogChange']
Number of predictors: 6
X_train shape: (2219, 6)
X_val shape:   (475, 6)
X_test shape:  (476, 6)


## Original tuning

Run this cell after the SPY data-preparation cell.

In [ ]:
# ============================================================
# ORIGINAL TUNING — SPY HYBRID ERNN A SELF-LAGGED
# Seven separately estimated nominal levels
# 200 Optuna trials per level
# ============================================================

import copy
import json
import time

import optuna
import torch.nn as nn
import torch.nn.functional as F
from optuna.samplers import TPESampler


MODEL_NAME = "spy_ernn_a_self"
FORM = "A"
LAG_TYPE = "self"
N_TRIALS = 200
MAX_EPOCHS = 200

TAU_LEVELS = [
    0.025,
    0.050,
    0.250,
    0.500,
    0.750,
    0.950,
    0.975,
]

OUTPUT_DIR = Path("spy_original_tuning_results") / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR.resolve())


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

def set_model_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------
# 2. AGD negative log-likelihood
# ------------------------------------------------------------

def hybrid_ernn_loss(y, expectile_forecast, sigma, tau):
    """Mean asymmetric-Gaussian negative log-likelihood."""

    eps = torch.finfo(expectile_forecast.dtype).eps
    sigma = torch.clamp(sigma, min=eps)

    tau_tensor = torch.as_tensor(
        tau,
        dtype=expectile_forecast.dtype,
        device=expectile_forecast.device,
    )

    difference = y - expectile_forecast

    asymmetric_squared_error = (
        torch.abs(
            tau_tensor
            - (difference < 0).to(expectile_forecast.dtype)
        )
        * difference.pow(2)
    )

    pi_tensor = torch.as_tensor(
        torch.pi,
        dtype=expectile_forecast.dtype,
        device=expectile_forecast.device,
    )

    normalising_constant = (
        torch.log(
            torch.sqrt(pi_tensor / tau_tensor)
            + torch.sqrt(pi_tensor / (1.0 - tau_tensor))
        )
        - torch.log(
            torch.as_tensor(
                2.0,
                dtype=expectile_forecast.dtype,
                device=expectile_forecast.device,
            )
        )
    )

    observation_nll = (
        torch.log(sigma)
        + normalising_constant
        + asymmetric_squared_error / sigma.pow(2)
    )

    return observation_nll.mean()


# ------------------------------------------------------------
# 3. CARE-hybrid recursion
# ------------------------------------------------------------

def compute_hybrid_location(
    y,
    X,
    gamma1,
    gamma2,
    gamma3,
    fnn,
    psi,
):
    """A-form self-lagged CARE-hybrid recursion."""

    fnn_output = fnn(X).squeeze(-1)
    zero_value = y.new_zeros(())

    care_forecasts = [zero_value]
    hybrid_forecasts = [zero_value]

    for time_index in range(1, len(y)):
        previous_return = y[time_index - 1]

        previous_state = care_forecasts[time_index - 1]
        positive_return = torch.clamp(previous_return, min=0)
        negative_return = torch.clamp(-previous_return, min=0)
        care_value = (
            gamma1 * previous_state
            + gamma2 * positive_return
            + gamma3 * negative_return
        )

        hybrid_value = (
            psi * care_value
            + (1.0 - psi) * fnn_output[time_index]
        )

        care_forecasts.append(care_value)
        hybrid_forecasts.append(hybrid_value)

    return (
        torch.stack(hybrid_forecasts),
        torch.stack(care_forecasts),
    )


# ------------------------------------------------------------
# 4. Hybrid ERNN model
# ------------------------------------------------------------

class HybridERNN_A_Self(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()

        self.eps = 1e-6

        self.gamma1_raw = nn.Parameter(
            torch.tensor(1.386)
        )

        self.gamma2_raw = nn.Parameter(
            torch.tensor(0.3)
        )

        self.gamma3_raw = nn.Parameter(
            torch.tensor(0.3)
        )
        self.psi_raw = nn.Parameter(
            torch.tensor(0.0)
        )

        self.sigma_raw = nn.Parameter(
            torch.tensor(0.5)
        )

        layers = []
        previous_dimension = input_dim

        for layer_index in range(config["n_hidden_layers"]):
            hidden_dimension = config[f"hidden_dim_{layer_index}"]

            layers.append(
                nn.Linear(previous_dimension, hidden_dimension)
            )

            if config["layernorm"]:
                layers.append(nn.LayerNorm(hidden_dimension))

            layers.append(
                self.get_activation(config["activation"])
            )

            if config["dropout"] > 0:
                layers.append(nn.Dropout(config["dropout"]))

            previous_dimension = hidden_dimension

        layers.append(nn.Linear(previous_dimension, 1))
        self.fnn = nn.Sequential(*layers)

        self.initialise_weights(config["weight_init"])

    @staticmethod
    def get_activation(name):
        return {
            "relu": nn.ReLU(),
            "elu": nn.ELU(),
            "gelu": nn.GELU(),
            "tanh": nn.Tanh(),
        }[name]

    def initialise_weights(self, method):
        for module in self.fnn.modules():
            if not isinstance(module, nn.Linear):
                continue

            if method == "glorot":
                nn.init.xavier_uniform_(module.weight)
            else:
                nn.init.kaiming_uniform_(module.weight)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, y, X):
        gamma1 = torch.sigmoid(self.gamma1_raw)
        gamma2 = F.softplus(self.gamma2_raw)

        gamma3 = F.softplus(
            self.gamma3_raw
        )
        psi = torch.sigmoid(self.psi_raw)
        sigma = F.softplus(self.sigma_raw) + self.eps

        hybrid_forecast, care_forecast = compute_hybrid_location(
            y=y,
            X=X,
            gamma1=gamma1,
            gamma2=gamma2,
            gamma3=gamma3,
            fnn=self.fnn,
            psi=psi,
        )

        return hybrid_forecast, sigma, care_forecast


# ------------------------------------------------------------
# 5. Original Bitcoin coverage-ratio ranking diagnostic
# ------------------------------------------------------------

def calculate_crp(y, forecast, tau):
    if tau <= 0.5:
        empirical_probability = (
            y <= forecast
        ).float().mean().item()
        target_probability = tau
    else:
        empirical_probability = (
            y > forecast
        ).float().mean().item()
        target_probability = 1.0 - tau

    crp = empirical_probability / target_probability
    crp_penalty = (crp - 1.0) ** 2

    return crp, crp_penalty, empirical_probability


# ------------------------------------------------------------
# 6. Optuna search space
# ------------------------------------------------------------

def suggest_ernn_config(trial):
    config = {
        "n_hidden_layers": trial.suggest_int(
            "n_hidden_layers", 1, 3
        ),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "elu", "gelu", "tanh"]
        ),
        "dropout": trial.suggest_float(
            "dropout", 0.0, 0.5
        ),
        "layernorm": trial.suggest_categorical(
            "layernorm", [True, False]
        ),
        "weight_init": trial.suggest_categorical(
            "weight_init", ["glorot", "he"]
        ),
        "lr": trial.suggest_float(
            "lr", 1e-4, 5e-2, log=True
        ),
        "optimizer": trial.suggest_categorical(
            "optimizer", ["adam", "adamw"]
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay", 0.0, 1e-4
        ),
        "gradient_clip": trial.suggest_categorical(
            "gradient_clip", [0.1, 1.0, 5.0]
        ),
        "patience": trial.suggest_categorical(
            "patience", [10, 20, 30]
        ),
    }

    for layer_index in range(config["n_hidden_layers"]):
        config[f"hidden_dim_{layer_index}"] = trial.suggest_int(
            f"hidden_dim_{layer_index}", 32, 256
        )

    return config


def make_optimizer(model, config):
    arguments = {
        "params": model.parameters(),
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
    }

    if config["optimizer"] == "adam":
        return torch.optim.Adam(**arguments)

    return torch.optim.AdamW(**arguments)


# ------------------------------------------------------------
# 7. Fit one trial configuration
# ------------------------------------------------------------

def train_ernn_configuration(config, tau, model_seed=BASE_SEED):
    set_model_seed(model_seed)

    model = HybridERNN_A_Self(
        input_dim=X_train.shape[1],
        config=config,
    ).to(device)

    optimizer = make_optimizer(model, config)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=10,
        factor=0.5,
        min_lr=1e-6,
    )

    best_validation_nll = np.inf
    best_model_state = None
    patience_counter = 0
    epochs_completed = 0

    for epoch in range(MAX_EPOCHS):
        epochs_completed = epoch + 1

        model.train()
        optimizer.zero_grad()

        train_forecast, train_sigma, _ = model(y_train, X_train)

        train_loss = hybrid_ernn_loss(
            y=y_train,
            expectile_forecast=train_forecast,
            sigma=train_sigma,
            tau=tau,
        )

        if not torch.isfinite(train_loss):
            return None

        train_loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            config["gradient_clip"],
        )

        optimizer.step()

        model.eval()

        with torch.no_grad():
            validation_forecast, validation_sigma, _ = model(
                y_val,
                X_val,
            )

            validation_loss = hybrid_ernn_loss(
                y=y_val,
                expectile_forecast=validation_forecast,
                sigma=validation_sigma,
                tau=tau,
            )

        if not torch.isfinite(validation_loss):
            return None

        scheduler.step(validation_loss)
        current_validation_nll = validation_loss.item()

        if current_validation_nll < best_validation_nll:
            best_validation_nll = current_validation_nll
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

            if patience_counter >= config["patience"]:
                break

    if best_model_state is None:
        return None

    model.load_state_dict(best_model_state)
    model.eval()

    with torch.no_grad():
        validation_forecast, validation_sigma, _ = model(y_val, X_val)

        validation_nll = hybrid_ernn_loss(
            y=y_val,
            expectile_forecast=validation_forecast,
            sigma=validation_sigma,
            tau=tau,
        ).item()

        (
            validation_crp,
            validation_crp_penalty,
            validation_hit_rate,
        ) = calculate_crp(
            y=y_val,
            forecast=validation_forecast,
            tau=tau,
        )

    parameters = {
        "gamma1": torch.sigmoid(model.gamma1_raw).item(),
        "gamma2": F.softplus(model.gamma2_raw).item(),
        "gamma3": F.softplus(model.gamma3_raw).item(),
        "psi": torch.sigmoid(model.psi_raw).item(),
        "sigma": (F.softplus(model.sigma_raw) + model.eps).item(),
    }

    return {
        "model": model,
        "validation_nll": validation_nll,
        "validation_crp": validation_crp,
        "validation_crp_penalty": validation_crp_penalty,
        "validation_hit_rate": validation_hit_rate,
        "epochs_completed": epochs_completed,
        **parameters,
    }


# ------------------------------------------------------------
# 8. Optuna objective and rank-based selection
# ------------------------------------------------------------

def objective_ernn(trial, tau):
    config = suggest_ernn_config(trial)
    result = train_ernn_configuration(config, tau)

    if result is None:
        raise optuna.TrialPruned(
            "Non-finite loss or no valid checkpoint."
        )

    for name in [
        "validation_nll",
        "validation_crp",
        "validation_crp_penalty",
        "validation_hit_rate",
        "epochs_completed",
        "gamma1",
        "gamma2",
        "gamma3",
        "psi",
        "sigma",
    ]:
        trial.set_user_attr(name, result[name])

    return result["validation_nll"]


def rank_completed_trials(study):
    completed_trials = [
        trial
        for trial in study.trials
        if (
            trial.state == optuna.trial.TrialState.COMPLETE
            and trial.value is not None
            and np.isfinite(trial.value)
            and "validation_crp_penalty" in trial.user_attrs
        )
    ]

    if not completed_trials:
        raise RuntimeError("No valid completed Optuna trials.")

    nll_order = sorted(
        completed_trials,
        key=lambda trial: trial.user_attrs["validation_nll"],
    )

    crp_order = sorted(
        completed_trials,
        key=lambda trial: trial.user_attrs[
            "validation_crp_penalty"
        ],
    )

    nll_ranks = {
        trial.number: rank
        for rank, trial in enumerate(nll_order, start=1)
    }

    crp_ranks = {
        trial.number: rank
        for rank, trial in enumerate(crp_order, start=1)
    }

    ranking_records = []

    for trial in completed_trials:
        rank_nll = nll_ranks[trial.number]
        rank_crp = crp_ranks[trial.number]

        ranking_records.append({
            "trial": trial.number,
            **trial.user_attrs,
            "rank_nll": rank_nll,
            "rank_crp": rank_crp,
            "sum_rank": rank_nll + rank_crp,
            **trial.params,
        })

    ranking_table = pd.DataFrame(ranking_records)

    ranking_table = ranking_table.sort_values(
        [
            "sum_rank",
            "validation_nll",
            "validation_crp_penalty",
            "trial",
        ]
    ).reset_index(drop=True)

    selected_trial_number = int(ranking_table.iloc[0]["trial"])

    selected_trial = next(
        trial
        for trial in completed_trials
        if trial.number == selected_trial_number
    )

    return selected_trial, ranking_table


# ------------------------------------------------------------
# 9. Run 200 trials at each nominal level
# ------------------------------------------------------------

best_configs_spy_ernn_a_self = {}
best_models_spy_ernn_a_self = {}
summary_records = []

overall_start_time = time.time()

for level_index, tau in enumerate(TAU_LEVELS):
    level_start_time = time.time()
    sampler_seed = BASE_SEED + level_index

    print("\n" + "=" * 72)
    print(f"Tuning {MODEL_NAME} at tau={tau:.3f}")
    print(f"Trials={N_TRIALS} | sampler seed={sampler_seed}")
    print("=" * 72)

    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=sampler_seed),
        study_name=f"{MODEL_NAME}_tau_{tau:.3f}",
    )

    study.optimize(
        lambda trial, current_tau=tau: objective_ernn(
            trial,
            current_tau,
        ),
        n_trials=N_TRIALS,
        show_progress_bar=True,
        gc_after_trial=True,
    )

    selected_trial, ranking_table = rank_completed_trials(study)
    tau_label = f"{tau:.3f}"

    ranking_table.insert(0, "tau", tau)

    ranking_path = OUTPUT_DIR / (
        f"{MODEL_NAME}_ranking_tau_{tau_label}.csv"
    )

    ranking_table.to_csv(ranking_path, index=False)

    selected_config = selected_trial.params.copy()

    selected_result = train_ernn_configuration(
        selected_config,
        tau,
    )

    if selected_result is None:
        raise RuntimeError(
            f"Selected trial could not be reproduced for tau={tau}."
        )

    selected_model = selected_result["model"]

    checkpoint_path = OUTPUT_DIR / (
        f"{MODEL_NAME}_best_tau_{tau_label}.pt"
    )

    torch.save(
        {
            "model_name": MODEL_NAME,
            "asset": "SPY",
            "form": FORM,
            "lag_type": LAG_TYPE,
            "tau": tau,
            "selected_trial": selected_trial.number,
            "config": selected_config,
            "model_state_dict": {
                key: value.detach().cpu()
                for key, value in selected_model.state_dict().items()
            },
            "validation_nll": selected_result["validation_nll"],
            "validation_crp": selected_result["validation_crp"],
            "validation_crp_penalty": selected_result[
                "validation_crp_penalty"
            ],
            "validation_hit_rate": selected_result[
                "validation_hit_rate"
            ],
            "estimated_parameters": {
                name: selected_result[name]
                for name in [
                    "gamma1",
                    "gamma2",
                    "gamma3",
                    "psi",
                    "sigma",
                ]
            },
            "feature_cols": feature_cols,
            "number_of_trials": N_TRIALS,
            "base_seed": BASE_SEED,
            "sampler_seed": sampler_seed,
        },
        checkpoint_path,
    )

    selected_record = {
        **selected_config,
        "selected_trial": selected_trial.number,
        **{
            name: selected_result[name]
            for name in [
                "validation_nll",
                "validation_crp",
                "validation_crp_penalty",
                "validation_hit_rate",
                "gamma1",
                "gamma2",
                "gamma3",
                "psi",
                "sigma",
            ]
        },
    }

    best_configs_spy_ernn_a_self[tau] = selected_record
    best_models_spy_ernn_a_self[tau] = selected_model

    runtime_hours = (
        time.time() - level_start_time
    ) / 3600

    summary_records.append({
        "tau": tau,
        "selected_trial": selected_trial.number,
        "validation_nll": selected_result["validation_nll"],
        "validation_crp": selected_result["validation_crp"],
        "validation_crp_penalty": selected_result[
            "validation_crp_penalty"
        ],
        "validation_hit_rate": selected_result[
            "validation_hit_rate"
        ],
        "gamma1": selected_result["gamma1"],
        "gamma2": selected_result["gamma2"],
        "gamma3": selected_result["gamma3"],
        "psi": selected_result["psi"],
        "sigma": selected_result["sigma"],
        "valid_trials": len(ranking_table),
        "runtime_hours": runtime_hours,
    })

    print("\nTop 10 rank-based trials:")
    print(
        ranking_table[[
            "trial",
            "validation_nll",
            "validation_crp",
            "validation_crp_penalty",
            "rank_nll",
            "rank_crp",
            "sum_rank",
        ]].head(10).to_string(index=False)
    )

    print("\nSelected trial:", selected_trial.number)
    print(
        "Validation NLL:",
        f"{selected_result['validation_nll']:.8f}",
    )
    print(
        "Validation CRP:",
        f"{selected_result['validation_crp']:.8f}",
    )
    print("Ranking saved to:", ranking_path)
    print("Checkpoint saved to:", checkpoint_path)


summary_spy_ernn_a_self = pd.DataFrame(summary_records)

summary_path = OUTPUT_DIR / f"{MODEL_NAME}_tuning_summary.csv"
summary_spy_ernn_a_self.to_csv(summary_path, index=False)

config_path = OUTPUT_DIR / f"{MODEL_NAME}_selected_configs.json"

with open(config_path, "w") as configuration_file:
    json.dump(
        {
            f"{tau:.3f}": config
            for tau, config in best_configs_spy_ernn_a_self.items()
        },
        configuration_file,
        indent=2,
    )

print("\n" + "=" * 72)
print("SPY ORIGINAL ERNN TUNING COMPLETED:", MODEL_NAME)
print("=" * 72)
display(summary_spy_ernn_a_self)

print(
    "Total runtime (hours):",
    f"{(time.time() - overall_start_time) / 3600:.3f}",
)
print("Summary saved to:", summary_path)
print("Selected configurations saved to:", config_path)


[I 2026-09-15 13:35:52,247] A new study created in memory with name: spy_ernn_a_self_tau_0.025


Model: spy_ernn_a_self
Output directory: /Users/miaomiaochen/Desktop/PhD program/Project 1/spy_original_tuning_results/spy_ernn_a_self

Tuning spy_ernn_a_self at tau=0.025
Trials=200 | sampler seed=2026


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-09-15 13:36:14,646] Trial 0 finished with value: 1.7633020877838135 and parameters: {'n_hidden_layers': 1, 'activation': 'elu', 'dropout': 0.49377524711883497, 'layernorm': False, 'weight_init': 'he', 'lr': 0.0006552020421842752, 'optimizer': 'adam', 'weight_decay': 8.940884610144999e-05, 'gradient_clip': 5.0, 'patience': 30, 'hidden_dim_0': 38}. Best is trial 0 with value: 1.7633020877838135.
[I 2026-09-15 13:36:38,950] Trial 1 finished with value: 0.726798951625824 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.07458356653667625, 'layernorm': False, 'weight_init': 'he', 'lr': 0.006969178688110181, 'optimizer': 'adam', 'weight_decay': 2.574006001095316e-05, 'gradient_clip': 1.0, 'patience': 30, 'hidden_dim_0': 102}. Best is trial 1 with value: 0.726798951625824.
[I 2026-09-15 13:36:41,549] Trial 2 finished with value: 1.8468976020812988 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.37112272198735796, 'layernorm': True, 'we

[I 2026-09-15 14:53:00,515] A new study created in memory with name: spy_ernn_a_self_tau_0.050


...

[I 2026-09-15 14:53:29,257] Trial 0 finished with value: 1.4234130382537842 and parameters: {'n_hidden_layers': 3, 'activation': 'relu', 'dropout': 0.05498225107357002, 'layernorm': True, 'weight_init': 'he', 'lr': 0.001007193483885585, 'optimizer': 'adam', 'weight_decay': 5.273500806799822e-05, 'gradient_clip': 5.0, 'patience': 20, 'hidden_dim_0': 146, 'hidden_dim_1': 251, 'hidden_dim_2': 82}. Best is trial 0 with value: 1.4234130382537842.
[I 2026-09-15 14:53:52,369] Trial 1 finished with value: 1.535031795501709 and parameters: {'n_hidden_layers': 1, 'activation': 'tanh', 'dropout': 0.025049491897366605, 'layernorm': True, 'weight_init': 'glorot', 'lr': 0.00015333527280501685, 'optimizer': 'adam', 'weight_decay': 1.7649100996182442e-05, 'gradient_clip': 0.1, 'patience': 30, 'hidden_dim_0': 55}. Best is trial 0 with value: 1.4234130382537842.
[I 2026-09-15 14:54:19,187] Trial 2 finished with value: -1.444385290145874 and parameters: {'n_hidden_layers': 2, 'activation': 'elu', 'dropou

[I 2026-09-15 16:48:20,475] A new study created in memory with name: spy_ernn_a_self_tau_0.250



Top 10 rank-based trials:
 trial  validation_nll  validation_crp  validation_crp_penalty  rank_nll  rank_crp  sum_rank
    82       -4.159088        1.473684                0.224377        16        16        32
   115       -4.181293        1.684211                0.468144         3        35        38
   159       -4.171156        1.642105                0.412299         9        31        40
   183       -4.175797        1.684211                0.468144         6        37        43
   190       -4.171140        1.726316                0.527535        10        44        54
   161       -4.173735        1.894737                0.800554         7        56        63
   172       -4.148421        1.726316                0.527535        25        42        67
    99       -4.156643        1.810526                0.656953        20        50        70
   192       -4.127475        1.684211                0.468144        34        38        72
   102       -4.119546        1.684211     

  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-09-15 16:48:44,609] Trial 0 finished with value: 0.9738381505012512 and parameters: {'n_hidden_layers': 1, 'activation': 'elu', 'dropout': 0.16735790150049712, 'layernorm': False, 'weight_init': 'glorot', 'lr': 0.00021817327615682693, 'optimizer': 'adamw', 'weight_decay': 1.8728142576828257e-05, 'gradient_clip': 1.0, 'patience': 30, 'hidden_dim_0': 153}. Best is trial 0 with value: 0.9738381505012512.
[I 2026-09-15 16:49:10,106] Trial 1 finished with value: -1.721933364868164 and parameters: {'n_hidden_layers': 2, 'activation': 'tanh', 'dropout': 0.16523551215002763, 'layernorm': True, 'weight_init': 'he', 'lr': 0.03513192693765672, 'optimizer': 'adam', 'weight_decay': 8.393147658894028e-05, 'gradient_clip': 1.0, 'patience': 20, 'hidden_dim_0': 151, 'hidden_dim_1': 164}. Best is trial 1 with value: -1.721933364868164.
[I 2026-09-15 16:49:34,230] Trial 2 finished with value: 0.22203324735164642 and parameters: {'n_hidden_layers': 1, 'activation': 'relu', 'dropout': 0.40467624618